# Day 019 Project — Eval Harness

Design a test set of **at least 5 test cases** for a factual Q&A use case and run the full eval harness.

## Requirements

1. Define `EVAL_CASES`: a list of at least 5 `TestCase` objects
2. At least 3 test cases must have `expected_keywords`
3. At least 1 test case must have `expected_answer` set (for judge scoring)
4. Call `run_eval(EVAL_CASES, system_prompt=SYSTEM_PROMPT)` to get results
5. Call `print_report(results)` to display the outcome
6. Run the scripted checks below

## Topic Ideas

| Topic | Example questions |
|-------|------------------|
| World capitals | What is the capital of Japan? |
| Basic science | What is the chemical symbol for water? |
| Programming | What does the `len()` function return in Python? |
| History | In what year did World War II end? |
| Math | What is the square root of 144? |

In [ ]:
import re, ollama
from dataclasses import dataclass, field

def exact_match(response: str, expected: str) -> bool:
    return response.strip().lower() == expected.strip().lower()

def contains_any(response: str, keywords: list[str]) -> bool:
    resp_lower = response.lower()
    return any(kw.lower() in resp_lower for kw in keywords)

JUDGE_PROMPT = """\
You are an evaluation judge. Score the response below on a scale of 1 to 5.

Question: {question}
Expected answer: {expected}
Actual response: {response}

Rubric:
1 = Completely wrong or irrelevant
2 = Mostly wrong with minor correct elements
3 = Partially correct but with significant gaps
4 = Mostly correct with minor issues
5 = Fully correct and complete

Respond with ONLY this format:
Score: <1-5>
Rationale: <one sentence>
"""

def llm_judge(question, response, expected, model='llama3.2'):
    prompt = JUDGE_PROMPT.format(question=question, expected=expected, response=response)
    raw = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    text = raw['message']['content']
    m = re.search(r'Score:\s*([1-5])', text)
    score = int(m.group(1)) if m else 3
    r = re.search(r'Rationale:\s*(.+)', text)
    rationale = r.group(1).strip() if r else text.strip()[:200]
    return {'score': score, 'rationale': rationale}

@dataclass
class TestCase:
    question: str
    expected_keywords: list[str] = field(default_factory=list)
    expected_answer: str = ''

@dataclass
class EvalResult:
    test_case: TestCase
    response: str
    passed: bool
    matched_keywords: list[str] = field(default_factory=list)
    judge_score: int = 0
    judge_rationale: str = ''

def run_eval(test_cases, system_prompt='You are a helpful assistant.',
             model='llama3.2', use_judge=False):
    results = []
    for tc in test_cases:
        raw = ollama.chat(model=model, messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': tc.question},
        ])
        response = raw['message']['content']
        matched = [kw for kw in tc.expected_keywords
                   if kw.lower() in response.lower()]
        passed = bool(matched) if tc.expected_keywords else True
        result = EvalResult(tc, response, passed, matched)
        if use_judge and tc.expected_answer:
            j = llm_judge(tc.question, response, tc.expected_answer, model)
            result.judge_score = j['score']
            result.judge_rationale = j['rationale']
        results.append(result)
    return results

def summarize_results(results):
    if not results:
        return {'total': 0, 'passed': 0, 'failed': 0,
                'pass_rate': 0.0, 'avg_judge_score': 0.0}
    total  = len(results)
    passed_count = sum(1 for r in results if r.passed)
    scores = [r.judge_score for r in results if r.judge_score > 0]
    return {
        'total': total,
        'passed': passed_count,
        'failed': total - passed_count,
        'pass_rate': round(passed_count / total, 4),
        'avg_judge_score': round(sum(scores) / len(scores), 2) if scores else 0.0,
    }

def print_report(results):
    s = summarize_results(results)
    print(f"Eval — {s['total']} cases | "
          f"Pass: {s['passed']}/{s['total']} ({s['pass_rate']*100:.1f}%)")
    if s['avg_judge_score'] > 0:
        print(f"  Avg judge score: {s['avg_judge_score']:.1f}/5")
    for i, r in enumerate(results, 1):
        icon = '✅' if r.passed else '❌'
        print(f"  {icon} {i}. {r.test_case.question[:55]}")
        if r.matched_keywords:
            print(f"     Keywords: {r.matched_keywords}")
        if r.judge_score:
            print(f"     Score {r.judge_score}/5 — {r.judge_rationale[:60]}")


## Your Test Set and Eval Run

In [ ]:
SYSTEM_PROMPT = 'You are a helpful and accurate assistant. Answer concisely.'

EVAL_CASES = [
    # TODO: add at least 5 TestCase objects
    # TestCase('Your question?', expected_keywords=['keyword'], expected_answer='Full answer'),
]

# Run the eval and print the report
# results = run_eval(EVAL_CASES, system_prompt=SYSTEM_PROMPT)
# print_report(results)


## Checks

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0

    # Check 1: all required components defined
    try:
        for name in ('exact_match', 'contains_any', 'llm_judge',
                     'TestCase', 'EvalResult', 'run_eval',
                     'summarize_results', 'print_report'):
            assert name in globals(), f'{name} not defined'
        passed += 1; print('✅ Check 1: all required components defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: EVAL_CASES has at least 5 cases
    try:
        assert 'EVAL_CASES' in globals(), 'EVAL_CASES not defined'
        assert len(EVAL_CASES) >= 5, f'Need at least 5 cases, got {len(EVAL_CASES)}'
        passed += 1; print(f'✅ Check 2: EVAL_CASES has {len(EVAL_CASES)} test cases (≥5 required)')
    except Exception as e:
        print(f'❌ Check 2: EVAL_CASES — {e}')

    # Check 3: at least 3 cases have expected_keywords
    try:
        with_keywords = [tc for tc in EVAL_CASES if tc.expected_keywords]
        assert len(with_keywords) >= 3, \
            f'Need at least 3 cases with keywords, got {len(with_keywords)}'
        passed += 1; print(f'✅ Check 3: {len(with_keywords)} cases have expected_keywords')
    except Exception as e:
        print(f'❌ Check 3: expected_keywords — {e}')

    # Check 4: run_eval works (one real Ollama call on a single test case)
    try:
        tc_test = TestCase('What is the capital of France?', expected_keywords=['paris'])
        old = sys.stdout; sys.stdout = io.StringIO()
        results_test = run_eval([tc_test])
        sys.stdout = old
        assert isinstance(results_test, list) and len(results_test) == 1
        assert isinstance(results_test[0], EvalResult)
        passed += 1; print('✅ Check 4: run_eval produces a valid EvalResult')
    except Exception as e:
        sys.stdout = old
        print(f'❌ Check 4: run_eval — {e}')

    # Check 5: summarize_results and print_report work on synthetic data
    try:
        tc_s = TestCase('Q?', expected_keywords=['yes'])
        synthetic = [
            EvalResult(tc_s, 'yes indeed', True, ['yes']),
            EvalResult(tc_s, 'no', False, []),
        ]
        s = summarize_results(synthetic)
        assert s['total'] == 2 and s['passed'] == 1
        assert abs(s['pass_rate'] - 0.5) < 1e-6
        old = sys.stdout; sys.stdout = io.StringIO()
        print_report(synthetic)
        out = sys.stdout.getvalue(); sys.stdout = old
        assert len(out) > 0
        passed += 1; print('✅ Check 5: summarize_results and print_report work')
    except Exception as e:
        sys.stdout = old
        print(f'❌ Check 5: summarize/report — {e}')

    if passed == total:
        print('🎉 Project complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()


## Bonus Challenges

- Run `run_eval(..., use_judge=True)` on the cases with `expected_answer` set and compare judge scores
- Try changing the `SYSTEM_PROMPT` (e.g., make it verbose vs. concise) and see if `pass_rate` changes
- Add a `contains_all(response, keywords)` metric (ALL keywords must appear) and use it for compound facts
- Save results to a JSON file so you can compare across runs: `json.dump([r.__dict__ ... for r in results], f)`
- Add a `negative_keywords` field to TestCase — fail if any negative keyword appears in the response